# Sequence refinement v004

This notebook preserves v003 and investigates the seven follow-up suggestions.
Only the approved Gym train/test files are used. Default mode reviews saved
results; `PA3_RUN_V004=1` executes and `PA3_RESUME_V004=1` reuses compatible
completed stages. No submission upload occurs.

Validated fitting continues to exclude original A validation breaths. B models
are freshly fit without B stopping feedback. The user's later request explicitly
authorizes a **separate final fit on all 67,905 labeled breaths** after selection.
That final checkpoint has no independent A/B holdout; earlier validation models
and their scores remain separate evidence.

The offline bidirectional GRU uses later known **input** values, never pressure
labels. All 80 input steps are provided for each test breath, and no streaming
restriction was found in the assignment/discussion PDFs. This is an offline task
assumption, not a claim that bidirectional inference is suitable for live control.

## Controlled candidates and source credits

- R/C already reach the current model at every step; skip duplicating that fix.
  Replace their numeric channels with learned 3-category × 2-dimensional tables
  in a separate representation test; the other 18 numeric channels retain
  training-only standardization.
- Match compact capacities: uni-GRU128×2 has 160,833 parameters; BiGRU80×2 has
  170,305; TCN72 has 161,561. Bidirectionality also changes future-input access.
- TCN: five residual blocks, two left-padded kernel-3 convolutions per block,
  dilations 1/2/4/8/16, timestep-wise channel LayerNorm, receptive field 125.
  This adapts the general architecture described by
  [Bai, Kolter and Koltun](https://arxiv.org/abs/1803.01271), using our own code.
  The [PyTorch GRU API](https://docs.pytorch.org/docs/2.15/generated/torch.nn.GRU.html)
  documents bidirectional outputs; execution retains the course's installed Torch.
- The old best checkpoint's actual LR is already 1e-5. Compare matched warm starts
  at 1e-5 and 1e-6 for 30 extra epochs, from identical weights/scalers, with fresh
  Adam and matched RNG in both arms. Old checkpoints did not save optimizer state,
  so this is a warm-start comparison, not exact optimizer-state resumption.
- Pairwise ensembles use A-only error patterns and weights in {.25,.5,.75};
  individual vertices remain candidates. Freeze members/weights before B.

Error contribution analysis and difficult-breath plots are in
`error_analysis_v004.ipynb`. High-MAE slices are ranked by their **total absolute
error share**, not MAE alone. No original Kaggle data/models or competition
solutions are used.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import gc,hashlib,json,time,copy,math
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from IPython.display import display
RUN=os.environ.get('PA3_RUN_V004','0')=='1'
RESUME=os.environ.get('PA3_RESUME_V004','0')=='1'
PROJECT=Path.cwd().resolve()
if not (PROJECT/'data').is_dir() and (PROJECT/'ventilator-pressure-prediction/data').is_dir():
    PROJECT=PROJECT/'ventilator-pressure-prediction'
OUT=PROJECT/'outputs/sequence_v004';EXP=OUT/'experiments';FINAL=OUT/'final_all'
PREV=PROJECT/'outputs/controlled_v003';DATA=PROJECT/'data'
SEED=20261102;MIN_GAIN=1e-6;THREADS=4
print('Mode:','execute refinements' if RUN else 'review saved refinements')
print('Artifacts:',OUT)
if RUN:import lightgbm as lgb

In [ ]:
STEPS = 80
DTYPES = {
    "id": "int64", "breath_id": "int32", "R": "int8", "C": "int8",
    "time_step": "float32", "u_in": "float32", "u_out": "int8",
    "pressure": "float32",
}


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def features(frame: pd.DataFrame) -> tuple[np.ndarray, list[str], np.ndarray]:
    """Return features in original row order, after verifying 80-row blocks."""
    if len(frame) % STEPS:
        raise ValueError("Unexpected breath length; do not reshape this file")
    blocks = frame["breath_id"].to_numpy().reshape(-1, STEPS)
    breath_ids = blocks[:, 0]
    if not np.all(blocks == breath_ids[:, None]):
        raise ValueError("Breaths are not contiguous 80-row blocks")
    if len(np.unique(breath_ids)) != len(breath_ids):
        raise ValueError("Repeated breath blocks")
    if not frame["id"].is_unique:
        raise ValueError("Duplicate row IDs")
    u = frame["u_in"].to_numpy().reshape(-1, STEPS)
    t = frame["time_step"].to_numpy().reshape(-1, STEPS)
    if not np.all(np.diff(t, axis=1) > 0):
        raise ValueError("Timestamps must increase inside each breath")
    for column in ("R", "C"):
        values = frame[column].to_numpy().reshape(-1, STEPS)
        if not np.all(values == values[:, :1]):
            raise ValueError(f"{column} changes within a breath")

    columns = {}
    for column in ("R", "C", "time_step", "u_in", "u_out"):
        columns[column] = frame[column].to_numpy(dtype=np.float32)
    step = np.arange(STEPS)
    dt = np.diff(t, axis=1, prepend=t[:, :1])
    cumulative = np.cumsum(u, axis=1, dtype=np.float32)
    columns["step"] = np.broadcast_to(step, u.shape).ravel()
    columns["dt"] = dt.ravel()
    columns["control_sum"] = cumulative.ravel()
    columns["control_mean"] = (cumulative / (step + 1)).ravel()
    # Valve opening is not measured flow: this is a history proxy, not volume.
    columns["control_integral"] = np.cumsum(u * dt, axis=1).ravel()
    for lag in (1, 2, 4, 8):
        shifted = np.empty_like(u)
        shifted[:, :lag] = u[:, :1]
        shifted[:, lag:] = u[:, :-lag]
        columns[f"u_in_lag_{lag}"] = shifted.ravel()
        columns[f"u_in_change_{lag}"] = (u - shifted).ravel()
    padded_sum = np.concatenate([np.zeros((len(u), 1), dtype=np.float32), cumulative], axis=1)
    for window in (4, 8):
        begin = np.maximum(0, step + 1 - window)
        columns[f"u_in_mean_{window}"] = (
            (padded_sum[:, step + 1] - padded_sum[:, begin]) / (step + 1 - begin)
        ).ravel()
    names = list(columns)
    matrix = np.column_stack(list(columns.values())).astype(np.float32, copy=False)
    if not np.isfinite(matrix).all() or "pressure" in names:
        raise ValueError("Invalid features")
    return matrix, names, breath_ids

In [ ]:
if RUN:
    import os
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

    import copy
    import hashlib
    import json
    import math
    import random
    import time
    import uuid
    from pathlib import Path

    import numpy as np
    import torch
    from torch import nn
    from torch.nn import functional as F
    from torch.utils.data import DataLoader, TensorDataset

    V4_R_VALUES = (5, 20, 50)
    V4_C_VALUES = (10, 20, 50)
    V4_INPUT_SHAPE = (80, 20)

    GRU_NUMERIC_SPEC = dict(family='gru', rc='numeric', hidden=128, layers=2,
                            dropout=.15, bidirectional=False)
    BIGRU_SPEC = dict(GRU_NUMERIC_SPEC, hidden=80, bidirectional=True)
    GRU_EMBEDDING_SPEC = dict(GRU_NUMERIC_SPEC, rc='embedding', embedding_dim=2)
    TCN_SPEC = dict(family='tcn', rc='numeric', channels=72, kernel_size=3,
                    dilations=[1, 2, 4, 8, 16], dropout=.10)
    DEFAULT_TRAIN_CFG = dict(lr=.0015, batch_size=256, max_epochs=100, patience=8,
                             min_delta=1e-6, max_seconds=600., min_lr=1e-5,
                             scheduler_factor=.5, scheduler_patience=1,
                             prediction_batch_size=512, gradient_clip=1.)


    def canonical_model_spec(model_spec):
        supplied = copy.deepcopy(dict(model_spec))
        family = str(supplied.get('family', 'gru')).lower()
        if family not in ('gru', 'tcn'):
            raise ValueError('family must be gru or tcn')
        rc = str(supplied.get('rc', 'numeric')).lower()
        if rc not in ('numeric', 'embedding'):
            raise ValueError('rc must be numeric or embedding')
        base = GRU_NUMERIC_SPEC if family == 'gru' else TCN_SPEC
        allowed = set(base) | {'embedding_dim'}
        unknown = set(supplied) - allowed
        if unknown:
            raise ValueError(f'Unknown model_spec fields: {sorted(unknown)}')
        spec = copy.deepcopy(base)
        spec.update(supplied)
        spec['family'], spec['rc'] = family, rc
        spec['dropout'] = float(spec['dropout'])
        if not 0. <= spec['dropout'] < 1.:
            raise ValueError('dropout must be in [0,1)')
        if family == 'gru':
            spec['hidden'], spec['layers'] = int(spec['hidden']), int(spec['layers'])
            spec['bidirectional'] = bool(spec['bidirectional'])
            if spec['hidden'] < 1 or spec['layers'] < 1:
                raise ValueError('GRU hidden/layers must be positive')
        else:
            spec['channels'], spec['kernel_size'] = int(spec['channels']), int(spec['kernel_size'])
            spec['dilations'] = [int(d) for d in spec['dilations']]
            if min(spec['channels'], spec['kernel_size']) < 1 or not spec['dilations'] or min(spec['dilations']) < 1:
                raise ValueError('TCN channels/kernel/dilations must be positive')
        if rc == 'embedding':
            spec['embedding_dim'] = int(spec.get('embedding_dim', 2))
            if spec['embedding_dim'] < 1:
                raise ValueError('embedding_dim must be positive')
        else:
            spec.pop('embedding_dim', None)
        return spec


    def seed_sequence(seed):
        random.seed(int(seed))
        np.random.seed(int(seed))
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
        torch.set_num_threads(4)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True)
        if hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
            torch.backends.cuda.matmul.allow_tf32 = False
        if hasattr(torch.backends.cudnn, 'allow_tf32'):
            torch.backends.cudnn.allow_tf32 = False


    def _sequence_device(device):
        value = torch.device(device)
        if value.type == 'cuda' and not torch.cuda.is_available():
            raise RuntimeError('CUDA requested but unavailable; use device=cpu only for toy checks')
        return value


    class _RCInput(nn.Module):
        def __init__(self, spec):
            super().__init__()
            self.rc = spec['rc']
            if self.rc == 'embedding':
                d = spec['embedding_dim']
                self.r_embedding = nn.Embedding(3, d)
                self.c_embedding = nn.Embedding(3, d)
                self.output_features = 18 + 2*d
            else:
                self.output_features = 20

        def forward(self, x):
            if self.rc == 'numeric':
                return x
            return torch.cat((self.r_embedding(x[..., 0].long()),
                              self.c_embedding(x[..., 1].long()), x[..., 2:]), dim=-1)


    class V4PressureGRU(nn.Module):
        def __init__(self, model_spec):
            super().__init__()
            spec = canonical_model_spec(model_spec)
            self.inputs = _RCInput(spec)
            self.gru = nn.GRU(self.inputs.output_features, spec['hidden'], num_layers=spec['layers'],
                              batch_first=True, bidirectional=spec['bidirectional'],
                              dropout=spec['dropout'] if spec['layers'] > 1 else 0.)
            width = spec['hidden'] * (2 if spec['bidirectional'] else 1)
            self.head = nn.Sequential(nn.Linear(width, 32), nn.SiLU(), nn.Linear(32, 1))

        def forward(self, x):
            sequence, _ = self.gru(self.inputs(x))
            return self.head(sequence).squeeze(-1)


    class _CausalConvUnit(nn.Module):
        def __init__(self, channels, kernel_size, dilation, dropout):
            super().__init__()
            self.left_padding = (kernel_size-1)*dilation
            self.conv = nn.Conv1d(channels, channels, kernel_size,
                                  dilation=dilation, padding=0)
            self.norm = nn.LayerNorm(channels)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x):
            # [B,C,T] -> normalize only C at each separate [B,T] location.
            y = self.conv(F.pad(x, (self.left_padding, 0)))
            y = self.norm(y.transpose(1, 2)).transpose(1, 2)
            return self.dropout(F.silu(y))


    class _CausalResidualBlock(nn.Module):
        def __init__(self, channels, kernel_size, dilation, dropout):
            super().__init__()
            self.first = _CausalConvUnit(channels, kernel_size, dilation, dropout)
            self.second = _CausalConvUnit(channels, kernel_size, dilation, dropout)

        def forward(self, x):
            return F.silu(x + self.second(self.first(x)))


    class V4PressureTCN(nn.Module):
        def __init__(self, model_spec):
            super().__init__()
            spec = canonical_model_spec(model_spec)
            self.inputs = _RCInput(spec)
            self.project = nn.Conv1d(self.inputs.output_features, spec['channels'], 1)
            self.blocks = nn.Sequential(*[
                _CausalResidualBlock(spec['channels'], spec['kernel_size'], d, spec['dropout'])
                for d in spec['dilations']])
            self.head = nn.Sequential(nn.Linear(spec['channels'], 32), nn.SiLU(), nn.Linear(32, 1))
            self.receptive_field = 1 + 2*(spec['kernel_size']-1)*sum(spec['dilations'])

        def forward(self, x):
            sequence = self.inputs(x).transpose(1, 2)
            sequence = self.blocks(self.project(sequence)).transpose(1, 2)
            return self.head(sequence).squeeze(-1)


    def build_model(model_spec):
        spec = canonical_model_spec(model_spec)
        return V4PressureGRU(spec) if spec['family'] == 'gru' else V4PressureTCN(spec)


    def model_parameter_count(model_spec):
        return int(sum(p.numel() for p in build_model(model_spec).parameters()))

In [ ]:
if RUN:
    def _raw_x(x):
        raw = np.asarray(x, dtype=np.float32)
        if raw.ndim != 3 or raw.shape[1:] != V4_INPUT_SHAPE or not len(raw):
            raise ValueError('Expected nonempty X[breaths,80,20]')
        if not np.isfinite(raw).all():
            raise ValueError('Nonfinite input feature')
        for column, values in ((0, V4_R_VALUES), (1, V4_C_VALUES)):
            if not np.isin(raw[..., column], values).all():
                raise ValueError('Unexpected physical R/C value')
            if not np.all(raw[..., column] == raw[:, :1, column]):
                raise ValueError('R/C must be constant within each breath')
        return raw


    def _training_arrays(x, y, u_out):
        raw = _raw_x(x)
        targets = np.asarray(y, dtype=np.float32)
        out = np.asarray(u_out)
        if targets.shape != raw.shape[:2] or out.shape != targets.shape:
            raise ValueError('X/Y/u_out shape mismatch')
        if not np.isfinite(targets).all() or not np.isin(out, [0, 1]).all():
            raise ValueError('Targets must be finite and u_out binary')
        if not np.any(out == 0):
            raise ValueError('No inspiratory targets')
        return raw, targets, out


    def fit_preprocessor(x_train, y_train, u_out_train, model_spec):
        """Fit exclusively on the caller-supplied training breaths; no validation access."""
        raw, y, out = _training_arrays(x_train, y_train, u_out_train)
        spec = canonical_model_spec(model_spec)
        numerical = raw if spec['rc'] == 'numeric' else raw[..., 2:]
        mean = numerical.mean(axis=(0, 1), dtype=np.float64).astype(np.float32)
        std = numerical.std(axis=(0, 1), dtype=np.float64).astype(np.float32)
        std[std < 1e-6] = 1.
        if spec['rc'] == 'embedding':
            mean = np.concatenate((np.zeros(2, np.float32), mean))
            std = np.concatenate((np.ones(2, np.float32), std))
        scored = y[out == 0]
        q25, q75 = np.quantile(scored, [.25, .75])
        return dict(input_shape=V4_INPUT_SHAPE, rc=spec['rc'],
                    x_mean=mean, x_std=std, target_center=float(np.median(scored)),
                    target_scale=max(float(q75-q25), 1.),
                    category_mappings=dict(R=list(V4_R_VALUES), C=list(V4_C_VALUES)),
                    fitted_training_breaths=int(len(raw)),
                    fitted_inspiratory_points=int((out == 0).sum()))


    def _validated_preprocessor(preprocess):
        prep = copy.deepcopy(preprocess)
        if tuple(prep.get('input_shape', V4_INPUT_SHAPE)) != V4_INPUT_SHAPE:
            raise ValueError('Checkpoint expects a different input shape')
        if prep.get('rc', 'numeric') not in ('numeric', 'embedding'):
            raise ValueError('Checkpoint has unknown R/C recipe')
        prep.setdefault('rc', 'numeric')
        prep['input_shape'] = V4_INPUT_SHAPE
        for name in ('x_mean', 'x_std'):
            prep[name] = np.array(prep[name], dtype=np.float32, copy=True)
            if prep[name].shape != (20,) or not np.isfinite(prep[name]).all():
                raise ValueError('Invalid checkpoint input scaler')
        if np.any(prep['x_std'] <= 0):
            raise ValueError('Invalid checkpoint standard deviations')
        if not np.isfinite(prep['target_center']) or not np.isfinite(prep['target_scale']) or prep['target_scale'] <= 0:
            raise ValueError('Invalid checkpoint target scaler')
        expected = dict(R=list(V4_R_VALUES), C=list(V4_C_VALUES))
        if prep.get('category_mappings', expected) != expected:
            raise ValueError('Unexpected checkpoint R/C category mapping')
        prep['category_mappings'] = expected
        return prep


    def transform_inputs(x, preprocess):
        raw = _raw_x(x)
        prep = _validated_preprocessor(preprocess)
        result = np.ascontiguousarray((raw-prep['x_mean'])/prep['x_std'], dtype=np.float32)
        if prep['rc'] == 'embedding':
            # Physical values are mapped directly to integer-valued codes, never standardized.
            for column, values in ((0, V4_R_VALUES), (1, V4_C_VALUES)):
                for index, value in enumerate(values):
                    result[..., column][raw[..., column] == value] = index
        if not np.isfinite(result).all():
            raise ValueError('Nonfinite standardized input')
        return result


    def inspiratory_mae(y, prediction, u_out):
        y, prediction, out = np.asarray(y), np.asarray(prediction), np.asarray(u_out)
        if y.shape != prediction.shape or out.shape != y.shape or not np.any(out == 0):
            raise ValueError('Metric shape/mask mismatch')
        if not np.isfinite(y).all() or not np.isfinite(prediction).all():
            raise ValueError('Nonfinite metric input')
        error = np.abs(prediction.astype(np.float64)-y.astype(np.float64))
        return float(error[out == 0].mean(dtype=np.float64))


    def _file_sha256(path):
        checksum = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024*1024), b''):
                checksum.update(block)
        return checksum.hexdigest()


    def _json_safe(value):
        if isinstance(value, dict):
            return {str(k): _json_safe(v) for k, v in value.items()}
        if isinstance(value, (tuple, list)):
            return [_json_safe(v) for v in value]
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, Path):
            return str(value)
        return value


    def _cpu_state(model):
        return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


    def _atomic_torch_save(payload, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
        try:
            torch.save(payload, temporary)
            os.replace(temporary, path)
        finally:
            if temporary.exists():
                temporary.unlink()


    def _new_output(path):
        path = Path(path)
        if path.exists():
            raise FileExistsError(f'Refusing to overwrite existing output: {path}')
        path.parent.mkdir(parents=True, exist_ok=True)
        return path


    def _epoch_event(path, value):
        if path is not None:
            with Path(path).open('a') as stream:
                stream.write(json.dumps(_json_safe(value), allow_nan=False)+'\n')
                stream.flush()


    def load_checkpoint(checkpoint_path, device='cpu'):
        """Load trusted v004 or old v002/v003 GRU state/scalers; return model, prep, payload."""
        path = Path(checkpoint_path)
        payload = torch.load(path, map_location='cpu', weights_only=False)
        if 'model_spec' in payload:
            spec = canonical_model_spec(payload['model_spec'])
            prep = _validated_preprocessor(payload['preprocess'])
        else:
            cfg = payload['config']
            spec = canonical_model_spec(dict(family='gru', rc='numeric', hidden=cfg['hidden'],
                layers=cfg['layers'], dropout=cfg['dropout'],
                bidirectional=any('_reverse' in k for k in payload['state_dict'])))
            if int(payload.get('n_features', 20)) != 20:
                raise ValueError('Legacy checkpoint does not use the required20features')
            prep = _validated_preprocessor(dict(input_shape=V4_INPUT_SHAPE, rc='numeric',
                x_mean=payload['x_mean'], x_std=payload['x_std'],
                target_center=payload['target_center'], target_scale=payload['target_scale']))
        if prep['rc'] != spec['rc']:
            raise ValueError('Model/scaler R/C recipes disagree')
        model = build_model(spec)
        model.load_state_dict(payload['state_dict'], strict=True)
        model.to(_sequence_device(device)).eval()
        return model, prep, payload


    @torch.no_grad()
    def _predict_native(model, x_preprocessed, preprocess, device, batch_size):
        model.eval()
        result = np.empty(x_preprocessed.shape[:2], dtype=np.float32)
        for begin in range(0, len(x_preprocessed), int(batch_size)):
            end = min(begin+int(batch_size), len(x_preprocessed))
            batch = torch.from_numpy(np.ascontiguousarray(x_preprocessed[begin:end])).to(device)
            result[begin:end] = model(batch).cpu().numpy()
        result = result*preprocess['target_scale']+preprocess['target_center']
        if not np.isfinite(result).all():
            raise RuntimeError('Nonfinite native-unit prediction')
        return result.astype(np.float32, copy=False)


    def predict_checkpoint(checkpoint_path, raw_x, device='cuda', batch_size=512):
        if int(batch_size) < 1:
            raise ValueError('batch_size must be positive')
        model, prep, _ = load_checkpoint(checkpoint_path, device=device)
        transformed = transform_inputs(raw_x, prep)
        return _predict_native(model, transformed, prep, _sequence_device(device), int(batch_size))

In [ ]:
if RUN:
    def _train_configuration(train_cfg, warm):
        cfg = dict(DEFAULT_TRAIN_CFG)
        if warm:
            cfg.update(max_epochs=30, max_seconds=None)
        unknown = set(train_cfg)-set(cfg)
        if unknown:
            raise ValueError(f'Unknown train_cfg fields: {sorted(unknown)}')
        cfg.update(dict(train_cfg))
        cfg['lr'] = float(cfg['lr'])
        cfg['batch_size'], cfg['max_epochs'] = int(cfg['batch_size']), int(cfg['max_epochs'])
        cfg['patience'] = int(cfg['patience'])
        cfg['prediction_batch_size'] = int(cfg['prediction_batch_size'])
        if min(cfg['batch_size'], cfg['max_epochs'], cfg['prediction_batch_size']) < 1:
            raise ValueError('Training counts/batch sizes must be positive')
        if not np.isfinite(cfg['lr']) or cfg['lr'] <= 0 or cfg['gradient_clip'] <= 0:
            raise ValueError('Learning rate and gradient clip must be positive')
        if not warm and (cfg['patience'] < 1 or cfg['min_lr'] < 0 or cfg['min_lr'] > cfg['lr']):
            raise ValueError('Invalid patience or learning-rate floor')
        if not 0 < cfg['scheduler_factor'] < 1 or int(cfg['scheduler_patience']) < 0:
            raise ValueError('Invalid scheduler configuration')
        if warm:
            if cfg['max_epochs'] > 30:
                raise ValueError('Matched warm-start investigation is bounded to at most30epochs')
            # Both matched arms run their requested epoch budget, without any stopping feedback.
            cfg['max_seconds'] = None
        elif cfg['max_seconds'] is not None and cfg['max_seconds'] <= 0:
            raise ValueError('max_seconds must be positive or None')
        return cfg


    def _prepare_stage(x, y, out, model_spec, start_checkpoint, device):
        spec = canonical_model_spec(model_spec)
        if start_checkpoint is None:
            prep = fit_preprocessor(x, y, out, spec)
            model = build_model(spec).to(device)
            source = None
        else:
            model, prep, payload = load_checkpoint(start_checkpoint, device=device)
            if 'model_spec' in payload:
                loaded_spec = canonical_model_spec(payload['model_spec'])
            else:
                old = payload['config']
                loaded_spec = canonical_model_spec(dict(family='gru', rc='numeric',
                    hidden=old['hidden'], layers=old['layers'], dropout=old['dropout'],
                    bidirectional=any('_reverse' in k for k in payload['state_dict'])))
            if loaded_spec != spec:
                raise ValueError('Warm start must preserve the exact architecture/R-C recipe')
            source = dict(path=str(Path(start_checkpoint)), sha256=_file_sha256(start_checkpoint),
                          format_version=int(payload.get('format_version', 0)),
                          source_best_epoch=payload.get('best_epoch'),
                          source_training_recipe=_json_safe(payload.get('training_recipe')))
        return model, prep, source


    def _training_loader(raw, targets, out, prep, cfg, seed, device):
        transformed = transform_inputs(raw, prep)
        scaled_targets = np.ascontiguousarray((targets-prep['target_center'])/prep['target_scale'], dtype=np.float32)
        masks = np.ascontiguousarray(out == 0)
        data = TensorDataset(torch.from_numpy(transformed), torch.from_numpy(scaled_targets),
                             torch.from_numpy(masks))
        return DataLoader(data, batch_size=cfg['batch_size'], shuffle=True, num_workers=0,
                          pin_memory=device.type == 'cuda',
                          generator=torch.Generator().manual_seed(int(seed)))


    def _training_epoch(model, loader, optimizer, cfg, prep, device):
        model.train()
        error, count = 0., 0
        for xb, yb, mask in loader:
            xb = xb.to(device, non_blocking=device.type == 'cuda')
            yb = yb.to(device, non_blocking=device.type == 'cuda')
            mask = mask.to(device, non_blocking=device.type == 'cuda')
            n = int(mask.sum().item())
            if n == 0:
                continue
            optimizer.zero_grad(set_to_none=True)
            absolute = (model(xb)-yb).abs().masked_select(mask)
            loss = absolute.mean()
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite masked L1 training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), float(cfg['gradient_clip']))
            optimizer.step()
            error += float(absolute.detach().sum().item())*prep['target_scale']
            count += n
        if count == 0:
            raise RuntimeError('No scored training points in an epoch')
        return float(error/count), count


    def _checkpoint_payload(model, prep, spec, cfg, seed, recipe, epoch, val_mae):
        return dict(format_version=4, model_spec=copy.deepcopy(spec), preprocess=copy.deepcopy(prep),
                    state_dict=_cpu_state(model), n_features=20,
                    parameter_count=int(sum(p.numel() for p in model.parameters())),
                    seed=int(seed), config=copy.deepcopy(cfg), best_epoch=int(epoch),
                    best_val_mae=None if val_mae is None else float(val_mae),
                    training_recipe=_json_safe(recipe))


    def fit_validate(x_train, y_train, u_out_train, x_val, y_val, u_out_val,
                     model_spec, train_cfg, seed, checkpoint_path, epoch_log_path=None,
                     start_checkpoint=None, device='cuda'):
        """A-only selection; returns a JSON-serializable report, never a retained model.

        Cold: train-only scalers, fresh Adam, plateau scheduler, best restoration.
        Warm: exact source scalers/weights, fresh Adam, constant LR, no early stopping,
              no scheduler; the unchanged source is eligible at extra epoch0.
        The caller must provide disjoint whole-breath train/validation groups.
        Epoch logs are JSONL: start event, epoch events, and a finished event.
        """
        began = time.monotonic()
        device = _sequence_device(device)
        raw, y, out = _training_arrays(x_train, y_train, u_out_train)
        val_raw, val_y, val_out = _training_arrays(x_val, y_val, u_out_val)
        warm = start_checkpoint is not None
        cfg, spec = _train_configuration(train_cfg, warm), canonical_model_spec(model_spec)
        checkpoint_path = _new_output(checkpoint_path)
        if epoch_log_path is not None:
            epoch_log_path = _new_output(epoch_log_path)
        seed_sequence(seed)
        model, prep, source = _prepare_stage(raw, y, out, spec, start_checkpoint, device)
        # Loading/building a model consumes RNG. Reset afterward so matched warm arms get
        # identical dropout draws and independent, identically seeded shuffle generators.
        if warm:
            seed_sequence(seed)
        loader = _training_loader(raw, y, out, prep, cfg, seed, device)
        val_z = transform_inputs(val_raw, prep)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
        scheduler = None if warm else torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=cfg['scheduler_factor'],
            patience=int(cfg['scheduler_patience']), min_lr=cfg['min_lr'])
        recipe = dict(mode='warm_validation' if warm else 'validation', validation_used=True,
                      optimizer='Adam', fresh_optimizer=True, Adam_betas=[.9,.999], Adam_eps=1e-8,
                      weight_decay=0., source_checkpoint=source,
                      scheduler='none_constant_lr' if warm else 'ReduceLROnPlateau',
                      scalers='source_exact' if warm else 'training_only',
                      train_breaths=int(len(raw)), validation_breaths=int(len(val_raw)))
        history, best_epoch, bad_epochs = [], 0, 0
        best_mae, best_state, initial_mae = float('inf'), None, None
        if warm:
            initial = _predict_native(model, val_z, prep, device, cfg['prediction_batch_size'])
            initial_mae = inspiratory_mae(val_y, initial, val_out)
            best_mae, best_state = initial_mae, _cpu_state(model)
            initial_payload = _checkpoint_payload(model, prep, spec, cfg, seed,
                dict(recipe, selected_stage_epochs=0, selected_lr_schedule=[]), 0, best_mae)
            _atomic_torch_save(initial_payload, checkpoint_path)
        _epoch_event(epoch_log_path, dict(event='start', seed=int(seed), model_spec=spec,
            train_cfg=cfg, recipe=recipe, initial_validation_mae=initial_mae))
        termination = 'epoch_cap'
        for epoch in range(1, cfg['max_epochs']+1):
            epoch_began = time.monotonic()
            lr_used = float(optimizer.param_groups[0]['lr'])
            train_mae, scored_count = _training_epoch(model, loader, optimizer, cfg, prep, device)
            prediction = _predict_native(model, val_z, prep, device, cfg['prediction_batch_size'])
            val_mae = inspiratory_mae(val_y, prediction, val_out)
            improved = val_mae < best_mae-float(cfg['min_delta'])
            if improved:
                best_mae, best_epoch, bad_epochs = val_mae, epoch, 0
                best_state = _cpu_state(model)
            else:
                bad_epochs += 1
            if scheduler is not None:
                scheduler.step(val_mae)
            record = dict(event='epoch', epoch=epoch, lr_used=lr_used,
                          lr_next=float(optimizer.param_groups[0]['lr']),
                          training_online_inspiratory_mae=train_mae,
                          training_scored_points=scored_count, validation_inspiratory_mae=val_mae,
                          best_val_mae=best_mae, best_epoch=best_epoch,
                          epoch_seconds=time.monotonic()-epoch_began,
                          elapsed_seconds=time.monotonic()-began)
            history.append(record)
            _epoch_event(epoch_log_path, record)
            if improved:
                interim = dict(recipe, selected_stage_epochs=best_epoch,
                    selected_lr_schedule=[r['lr_used'] for r in history[:best_epoch]])
                _atomic_torch_save(_checkpoint_payload(model, prep, spec, cfg, seed,
                    interim, best_epoch, best_mae), checkpoint_path)
            if not warm and bad_epochs >= cfg['patience']:
                termination = 'early_stopping'
                break
            if not warm and cfg['max_seconds'] is not None and time.monotonic()-began >= cfg['max_seconds']:
                termination = 'time_cap'
                break
        if best_state is None:
            raise RuntimeError('No finite validation checkpoint was produced')
        model.load_state_dict(best_state, strict=True)
        model.eval()
        restored = _predict_native(model, val_z, prep, device, cfg['prediction_batch_size'])
        restored_mae = inspiratory_mae(val_y, restored, val_out)
        if not np.isclose(restored_mae, best_mae, rtol=1e-6, atol=1e-6):
            raise RuntimeError('Restored checkpoint metric differs from selected best')
        selected_rates = [r['lr_used'] for r in history[:best_epoch]]
        final_recipe = dict(recipe, selected_stage_epochs=best_epoch,
                            selected_lr_schedule=selected_rates, executed_epochs=len(history),
                            executed_lr_schedule=[r['lr_used'] for r in history],
                            termination=termination)
        _atomic_torch_save(_checkpoint_payload(model, prep, spec, cfg, seed,
            final_recipe, best_epoch, restored_mae), checkpoint_path)
        report = dict(model_spec=spec, train_cfg=cfg, seed=int(seed),
            parameter_count=int(sum(p.numel() for p in model.parameters())),
            best_epoch=best_epoch, validation_mae=restored_mae,
            initial_validation_mae=initial_mae, executed_epochs=len(history),
            selected_lr_schedule=selected_rates, epochs=history,
            termination=termination, elapsed_seconds=time.monotonic()-began,
            training_breaths=int(len(raw)), validation_breaths=int(len(val_raw)),
            scored_validation_points=int((val_out == 0).sum()),
            checkpoint_path=str(checkpoint_path), checkpoint_sha256=_file_sha256(checkpoint_path),
            preprocessing=_json_safe(prep), training_recipe=final_recipe,
            validation_used=True, best_checkpoint_restored=True)
        _epoch_event(epoch_log_path, dict(event='finished', best_epoch=best_epoch,
            validation_mae=restored_mae, termination=termination,
            checkpoint_sha256=report['checkpoint_sha256']))
        return _json_safe(report)

In [ ]:
if RUN:
    def fit_fixed(x_train, y_train, u_out_train, model_spec, train_cfg, seed,
                  lr_schedule, checkpoint_path, start_checkpoint=None,
                  epoch_log_path=None, device='cuda'):
        """Fit a frozen A-selected recipe without accepting validation inputs or labels.

        Cold fits its own training-only scalers. Warm retains exact source scalers and
        resets Adam. Every supplied LR is used for one full epoch; the final state is
        saved without validation-based selection, early stopping, or a time cap.
        An empty schedule is valid only for a warm zero-extra-epoch selection.
        """
        began = time.monotonic()
        device = _sequence_device(device)
        raw, y, out = _training_arrays(x_train, y_train, u_out_train)
        warm = start_checkpoint is not None
        cfg, spec = _train_configuration(train_cfg, warm), canonical_model_spec(model_spec)
        rates = [float(rate) for rate in lr_schedule]
        if not all(np.isfinite(rate) and rate > 0 for rate in rates):
            raise ValueError('Every replayed learning rate must be positive and finite')
        if not rates and not warm:
            raise ValueError('A cold fixed fit requires at least one frozen training epoch')
        if warm and len(rates) > 30:
            raise ValueError('Warm replay exceeds the matched30epoch investigation')
        checkpoint_path = _new_output(checkpoint_path)
        if epoch_log_path is not None:
            epoch_log_path = _new_output(epoch_log_path)
        seed_sequence(seed)
        model, prep, source = _prepare_stage(raw, y, out, spec, start_checkpoint, device)
        if warm:
            seed_sequence(seed)
        loader = _training_loader(raw, y, out, prep, cfg, seed, device)
        optimizer = torch.optim.Adam(model.parameters(), lr=rates[0] if rates else cfg['lr'])
        recipe = dict(mode='warm_fixed' if warm else 'fixed', validation_used=False,
                      selected_using='A only; budget supplied by caller',
                      optimizer='Adam', fresh_optimizer=True, Adam_betas=[.9,.999], Adam_eps=1e-8,
                      weight_decay=0., source_checkpoint=source, scheduler='replayed_A_schedule',
                      scalers='source_exact' if warm else 'training_only',
                      train_breaths=int(len(raw)), selected_stage_epochs=len(rates),
                      selected_lr_schedule=rates)
        history = []
        _epoch_event(epoch_log_path, dict(event='start', model_spec=spec, train_cfg=cfg,
            seed=int(seed), recipe=recipe))
        for epoch, rate in enumerate(rates, 1):
            epoch_began = time.monotonic()
            for group in optimizer.param_groups:
                group['lr'] = rate
            train_mae, scored_count = _training_epoch(model, loader, optimizer, cfg, prep, device)
            record = dict(event='epoch', epoch=epoch, lr_used=rate,
                          lr_next=rates[epoch] if epoch < len(rates) else None,
                          training_online_inspiratory_mae=train_mae,
                          training_scored_points=scored_count, validation_used=False,
                          epoch_seconds=time.monotonic()-epoch_began,
                          elapsed_seconds=time.monotonic()-began)
            history.append(record)
            _epoch_event(epoch_log_path, record)
            # Save each completed epoch so interruption preserves concrete progress.
            progress_recipe = dict(recipe, completed_epochs=epoch, completed=False)
            _atomic_torch_save(_checkpoint_payload(model, prep, spec, cfg, seed,
                progress_recipe, epoch, None), checkpoint_path)
        model.eval()
        recipe.update(completed_epochs=len(rates), completed=True)
        _atomic_torch_save(_checkpoint_payload(model, prep, spec, cfg, seed,
            recipe, len(rates), None), checkpoint_path)
        report = dict(model_spec=spec, train_cfg=cfg, seed=int(seed),
            parameter_count=int(sum(p.numel() for p in model.parameters())),
            executed_epochs=len(rates), selected_lr_schedule=rates, epochs=history,
            termination='fixed_A_budget', elapsed_seconds=time.monotonic()-began,
            training_breaths=int(len(raw)), checkpoint_path=str(checkpoint_path),
            checkpoint_sha256=_file_sha256(checkpoint_path),
            preprocessing=_json_safe(prep), training_recipe=recipe, validation_used=False)
        _epoch_event(epoch_log_path, dict(event='finished', executed_epochs=len(rates),
            validation_used=False, checkpoint_sha256=report['checkpoint_sha256']))
        return _json_safe(report)

In [ ]:
BASE_SPEC=dict(family='gru',rc='numeric',hidden=128,layers=2,dropout=.15,bidirectional=False)
BI_SPEC=dict(BASE_SPEC,hidden=80,bidirectional=True)
EMBED_SPEC=dict(BASE_SPEC,rc='embedding',embedding_dim=2)
CONV_SPEC=dict(family='tcn',rc='numeric',channels=72,kernel_size=3,dilations=[1,2,4,8,16],dropout=.10)
CFG=dict(lr=.0015,batch_size=256,max_epochs=100,patience=8,min_delta=1e-6,
         max_seconds=600.,min_lr=1e-5,scheduler_factor=.5,scheduler_patience=1,
         prediction_batch_size=512,gradient_clip=1.)
CANDIDATES=[
 ('warm_current',dict(kind='warm',model_spec=BASE_SPEC,train_cfg=dict(CFG,lr=1e-5,max_epochs=30,min_lr=0.),seed=SEED,base='gru_s20261102')),
 ('warm_tenth',dict(kind='warm',model_spec=BASE_SPEC,train_cfg=dict(CFG,lr=1e-6,max_epochs=30,min_lr=0.),seed=SEED,base='gru_s20261102')),
 ('bigru80',dict(kind='fresh',model_spec=BI_SPEC,train_cfg=CFG,seed=SEED)),
 ('gru_embedding',dict(kind='fresh',model_spec=EMBED_SPEC,train_cfg=CFG,seed=SEED)),
 ('tcn_lr15',dict(kind='fresh',model_spec=CONV_SPEC,train_cfg=CFG,seed=SEED)),
 ('tcn_lr30',dict(kind='fresh',model_spec=CONV_SPEC,train_cfg=dict(CFG,lr=.003),seed=SEED)),
]
PLAN=dict(version=4,seed=SEED,candidates=CANDIDATES,weights_grid=[.25,.5,.75],
    reference='v003 best A individual, seed20261102',
    B_rule='Freeze A recipes/budgets/weights before scoring B. New individual must improve reference on A and B; pair must improve best individual on A and B and reference on B; choose lowest A MAE eligible.',
    final_all_labeled_authorized=True,final_training_breaths=67905,
    final_rule='Fresh all-labeled scalers and weights; replay A-selected epochs/LRs; warm recipes reset Adam after the frozen base; no validation feedback or new holdout score',
    offline_bidirectional=True,pressure_is_never_an_input=True)
def signature(value):return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+'.tmp')
    temporary.write_text(json.dumps(value,indent=2)+'\n');os.replace(temporary,path)
def event(kind,**values):
    with (EXP/'events.jsonl').open('a') as f:f.write(json.dumps(dict(timestamp_utc=datetime.now(timezone.utc).isoformat(),event=kind,**values))+'\n')
if RUN:
    # The original batch must finish first; no duplicate GPU training is launched.
    previous_selection=json.loads((PREV/'selected_approach.json').read_text())
    assert (PREV/'submission.csv').is_file() and previous_selection['uploaded']==False
    EXP.mkdir(parents=True,exist_ok=RESUME)
    quality=json.loads((PROJECT/'quality_audit_v002.json').read_text())
    assert digest(DATA/'train.csv')==quality['files']['train.csv']['sha256']
    frame=pd.read_csv(DATA/'train.csv',dtype=DTYPES)
    flat,names,breath_ids=features(frame);X=flat.reshape(-1,80,len(names));del flat
    Y=frame.pressure.to_numpy().reshape(-1,80);UOUT=frame.u_out.to_numpy().reshape(-1,80)
    RC=frame[['R','C']].to_numpy()[::80];ROW_IDS=frame.id.to_numpy().reshape(-1,80)
    saved=np.load(PREV/'split_breath_ids.npz');index=pd.Index(breath_ids)
    GROUPS={key:np.sort(index.get_indexer(saved[key])) for key in ['A_fit','A_validation','B_fit','B_validation']}
    assert all((v>=0).all() for v in GROUPS.values())
    assert len(GROUPS['A_fit'])==61114 and len(GROUPS['B_fit'])==54323
    assert len(GROUPS['A_validation'])==len(GROUPS['B_validation'])==6791
    for group in ['A_fit','B_fit']:assert not np.intersect1d(GROUPS[group],GROUPS['A_validation']).size
    assert not np.intersect1d(GROUPS['B_fit'],GROUPS['B_validation']).size
    ALL=np.arange(len(breath_ids));assert len(ALL)==67905
    nb_source=json.loads((PROJECT/'sequence_refinement_v004.ipynb').read_text())
    source_sha=hashlib.sha256('\n'.join(''.join(c['source']) for c in nb_source['cells'] if c['cell_type']=='code').encode()).hexdigest()
    design=json.loads(json.dumps(dict(plan=PLAN,features=names,source_sha256=source_sha,
        train_sha256=digest(DATA/'train.csv'),previous_design_sha256=digest(PREV/'design.json'),
        group_hashes={k:hashlib.sha256(breath_ids[v].tobytes()).hexdigest() for k,v in GROUPS.items()})))
    if RESUME:assert json.loads((EXP/'design.json').read_text())==design,'Changed source/plan/data; use a new run'
    else:
        write_json(EXP/'design.json',design)
        np.savez_compressed(EXP/'split_breath_ids.npz',**{k:breath_ids[v] for k,v in GROUPS.items()},final_all=breath_ids)
    event('initialized',resume=RESUME,source_sha256=source_sha)
    del frame;gc.collect()

## Reuse validated references and make every stage explicit

Legacy GRU checkpoints are loaded with exactly their original feature/target
scalers. B reference models are the B-only fits from v003, not A weights. Every
new checkpoint records its exact source hash, model recipe, fit-group hash and
prediction order. Completed stages are reused only if identity and hashes match.
Interrupted incomplete files are archived before a deterministic stage restart.

In [ ]:
def target_context(split):
    indices=GROUPS[split+'_validation'];mask=UOUT[indices]==0
    return indices,mask,Y[indices][mask],ROW_IDS[indices][mask]
def save_score(tag,prediction,split):
    ids,mask,y,row_ids=target_context(split)
    p=np.asarray(prediction,dtype=np.float64)
    assert p.shape==y.shape and np.isfinite(p).all()
    np.save(EXP/(tag+'_predictions.npy'),p)
    np.save(EXP/(split+'_row_ids.npy'),row_ids)
    rc=np.repeat(RC[ids],80,axis=0)[mask.ravel()]
    e=np.abs(p-y)
    table=pd.DataFrame({'R':rc[:,0],'C':rc[:,1],'absolute_error':e,'signed_error':p-y})
    table.groupby(['R','C']).agg(count=('absolute_error','size'),mae=('absolute_error','mean'),bias=('signed_error','mean'),total_absolute_error=('absolute_error','sum')).to_csv(EXP/(tag+'_by_rc.csv'))
    return float(e.mean()),hashlib.sha256(p.tobytes()).hexdigest()
def refresh_results():
    rows=[]
    for p in EXP.glob('record_*.json'):
        r=json.loads(p.read_text());rows.append({k:r.get(k) for k in ['name','split','kind','validation_mae','elapsed_seconds','best_epoch','training_breaths','parameter_count','checkpoint_path']})
    pd.DataFrame(rows).to_csv(EXP/'results.csv',index=False)
def old_record(trial):return json.loads((PREV/(trial+'.json')).read_text())
def old_ref(name,trial,kind,seed):
    old=old_record(trial)
    return dict(name=name,kind=kind,seed=seed,old_A_trial=trial,model_spec=BASE_SPEC if kind=='reference_sequence' else None,
                train_cfg=CFG if kind=='reference_sequence' else None)
def old_trial_for_B(recipe):
    if recipe['kind']=='reference_sequence':return 'B_member_s'+str(recipe['seed'])
    if recipe['old_A_trial']=='full_incumbent':return 'B_incumbent'
    return None
def old_rates(record):
    history=json.loads(Path(record['epoch_log_file']).read_text())['epochs']
    n=int(record['best_epoch']);return [record['definition']['params']['lr']]+[r['lr'] for r in history[:n-1]]
def archive_partial(*paths):
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    for p in paths:
        p=Path(p)
        if p.exists():p.rename(p.with_name(p.name+'.interrupted-'+stamp))
def infer_record(record,raw):
    if record['kind']=='reference_tree':
        model=lgb.Booster(model_file=record['checkpoint_path'])
        return model.predict(raw.reshape(-1,20),num_threads=THREADS).reshape(raw.shape[:2])
    return predict_checkpoint(record['checkpoint_path'],raw)
def record_reference(recipe,split):
    name=recipe['name'];tag=split+'_'+name
    which=recipe['old_A_trial'] if split=='A' else old_trial_for_B(recipe)
    if which is None:return None
    old=old_record(which)
    assert old['split']==split
    assert old['identity']['fit_breath_sha256']==design['group_hashes'][split+'_fit']
    checkpoint=Path(old['model_file']);assert digest(checkpoint)==old['model_sha256']
    prediction=np.load(old['prediction_file']);score,pred_hash=save_score(tag,prediction,split)
    assert np.isclose(score,old['validation_mae'],atol=1e-9)
    r=dict(name=name,split=split,kind=recipe['kind'],recipe=recipe,validation_mae=score,
        prediction_file=str(EXP/(tag+'_predictions.npy')),prediction_sha256=pred_hash,
        checkpoint_path=str(checkpoint),checkpoint_sha256=digest(checkpoint),elapsed_seconds=0.,
        best_epoch=old.get('best_epoch'),training_breaths=old['training_breaths'],
        parameter_count=160833 if recipe['kind']=='reference_sequence' else None,
        selected_lr_schedule=old_rates(old) if split=='A' and recipe['kind']=='reference_sequence' else None,
        old_metadata=old,fit_group_sha256=design['group_hashes'][split+'_fit'])
    write_json(EXP/('record_'+tag+'.json'),r);refresh_results();return r
def fit_A(name,recipe):
    tag='A_'+name;path=EXP/('record_'+tag+'.json')
    identity=signature(dict(recipe=recipe,design=signature(design),split='A'))
    if path.exists():
        r=json.loads(path.read_text());assert r['identity']==identity
        assert digest(Path(r['checkpoint_path']))==r['checkpoint_sha256'];return r
    fit_ids=GROUPS['A_fit'];val_ids,mask,y,_=target_context('A')
    checkpoint=EXP/(tag+'.pt');epoch_log=EXP/(tag+'_epochs.jsonl')
    archive_partial(checkpoint,epoch_log)
    base=A_RECORDS[recipe['base']]['checkpoint_path'] if recipe['kind']=='warm' else None
    event('stage_started',stage=tag,recipe=recipe,source_checkpoint=base)
    report=fit_validate(X[fit_ids],Y[fit_ids],UOUT[fit_ids],X[val_ids],Y[val_ids],UOUT[val_ids],
        recipe['model_spec'],recipe['train_cfg'],recipe['seed'],checkpoint,epoch_log_path=epoch_log,start_checkpoint=base)
    prediction=predict_checkpoint(checkpoint,X[val_ids])[mask]
    score,pred_hash=save_score(tag,prediction,'A')
    assert np.isclose(score,report['validation_mae'],rtol=1e-6,atol=1e-6)
    r=dict(name=name,split='A',kind=recipe['kind'],recipe=recipe,identity=identity,
        validation_mae=score,prediction_file=str(EXP/(tag+'_predictions.npy')),prediction_sha256=pred_hash,
        checkpoint_path=str(checkpoint),checkpoint_sha256=digest(checkpoint),elapsed_seconds=report['elapsed_seconds'],
        best_epoch=report['best_epoch'],selected_lr_schedule=report['selected_lr_schedule'],
        training_breaths=len(fit_ids),parameter_count=report['parameter_count'],report=report,
        fit_group_sha256=design['group_hashes']['A_fit'])
    write_json(path,r);refresh_results();event('stage_completed',stage=tag,validation_mae=score,seconds=r['elapsed_seconds'])
    gc.collect();torch.cuda.empty_cache();return r

if RUN:
    REFERENCE_RECIPES=[old_ref('gru_s20261102','full_screen_gru_lr_0015','reference_sequence',20261102),
        old_ref('gru_s20261103','A_member_s20261103','reference_sequence',20261103),
        old_ref('gru_s20261104','A_member_s20261104','reference_sequence',20261104),
        old_ref('tree_511','full_screen_leaves_511','reference_tree',SEED),
        old_ref('tree_5000','full_screen_trees_5000','reference_tree',SEED)]
    A_RECORDS={r['name']:record_reference(r,'A') for r in REFERENCE_RECIPES}
    assert np.isclose(A_RECORDS['gru_s20261102']['validation_mae'],.4962725305798752,atol=1e-8)
    # Verify installed CUDA kernels and legacy inference before long new fits.
    seed_sequence(SEED)
    toy=np.zeros((2,80,20),np.float32);toy[:,:,0]=5;toy[:,:,1]=10
    for s in [BASE_SPEC,BI_SPEC,EMBED_SPEC,CONV_SPEC]:
        m=build_model(s).cuda();z=torch.zeros((2,80,20),device='cuda')
        out=m(z);assert out.shape==(2,80) and torch.isfinite(out).all()
        out.mean().backward();del m,z,out
    torch.cuda.empty_cache();event('CUDA_component_preflight_passed')

## warm_current

In [ ]:
if RUN:
    name,recipe=CANDIDATES[0]
    A_RECORDS[name]=fit_A(name,recipe)

## warm_tenth

In [ ]:
if RUN:
    name,recipe=CANDIDATES[1]
    A_RECORDS[name]=fit_A(name,recipe)

## bigru80

In [ ]:
if RUN:
    name,recipe=CANDIDATES[2]
    A_RECORDS[name]=fit_A(name,recipe)

## gru_embedding

In [ ]:
if RUN:
    name,recipe=CANDIDATES[3]
    A_RECORDS[name]=fit_A(name,recipe)

## tcn_lr15

In [ ]:
if RUN:
    name,recipe=CANDIDATES[4]
    A_RECORDS[name]=fit_A(name,recipe)

## tcn_lr30

In [ ]:
if RUN:
    name,recipe=CANDIDATES[5]
    A_RECORDS[name]=fit_A(name,recipe)

## Error diversity and A-only ensemble selection

Use aligned native-unit A predictions. Signed-residual correlation describes
similarity, but actual masked MAE decides usefulness. Search only pairs with fixed
weights .25/.5/.75, retain individual vertices, and freeze the result before B.
This is a bounded validation search, not proof of an optimal ensemble.

In [ ]:
if RUN:
    all_a=list(A_RECORDS.values());unique=[];seen=set()
    for r in all_a:
        if r['prediction_sha256'] not in seen:
            unique.append(r);seen.add(r['prediction_sha256'])
    _,_,a_y,_=target_context('A')
    a_predictions={r['name']:np.load(r['prediction_file']) for r in unique}
    labels=[r['name'] for r in unique]
    residuals=np.stack([a_predictions[n]-a_y for n in labels])
    pd.DataFrame(np.corrcoef(residuals),index=labels,columns=labels).to_csv(EXP/'A_residual_correlations.csv')
    pairs=[]
    for i,first in enumerate(labels):
        for second in labels[i+1:]:
            for weight in PLAN['weights_grid']:
                pred=weight*a_predictions[first]+(1-weight)*a_predictions[second]
                pairs.append(dict(first=first,second=second,weight_first=weight,weight_second=1-weight,
                    validation_mae=float(np.abs(pred-a_y).mean())))
    pair_table=pd.DataFrame(pairs).sort_values(['validation_mae','first','second','weight_first'])
    pair_table.to_csv(EXP/'A_pair_search.csv',index=False)
    best_individual=min(unique,key=lambda r:(r['validation_mae'],r['name']))
    pair=pair_table.iloc[0].to_dict()
    frozen=dict(best_individual=best_individual['name'],best_individual_A_mae=best_individual['validation_mae'],
        pair_members=[pair['first'],pair['second']],pair_weights=[pair['weight_first'],pair['weight_second']],
        pair_A_mae=pair['validation_mae'],reference='gru_s20261102',
        model_hashes={r['name']:r['checkpoint_sha256'] for r in unique},
        rule=PLAN['B_rule'],weight_selection='A only',B_used_to_tune_weights=False)
    if (EXP/'locked_before_B.json').exists():assert json.loads((EXP/'locked_before_B.json').read_text())==frozen
    else:write_json(EXP/'locked_before_B.json',frozen)
    event('A_selection_locked',selection=frozen)
    display(pair_table.head(10))

## Fresh B confirmation, without B stopping feedback

Reuse B-only reference models when available. Fresh models fit Bfit only, replaying
A-selected LR schedules. A warm candidate starts from its **B-only base**, retaining
that base's scalers. Starting B fine-tuning from an A checkpoint would leak B labels;
this code never does that. B is repeatedly viewed resampling confirmation, not a
newly untouched final test.

In [ ]:
def fit_B(name):
    if name in B_RECORDS:return B_RECORDS[name]
    a=A_RECORDS[name];recipe=a['recipe'];tag='B_'+name;path=EXP/('record_'+tag+'.json')
    identity=signature(dict(recipe=recipe,A_hash=a['checkpoint_sha256'],rates=a.get('selected_lr_schedule'),design=signature(design),split='B'))
    if path.exists():
        r=json.loads(path.read_text());assert r.get('identity')==identity
        assert digest(Path(r['checkpoint_path']))==r['checkpoint_sha256'];B_RECORDS[name]=r;return r
    if recipe['kind'].startswith('reference'):
        ref=record_reference(recipe,'B')
        if ref is not None:
            ref['identity']=identity;write_json(path,ref);B_RECORDS[name]=ref;return ref
    fit_ids=GROUPS['B_fit'];val_ids,mask,y,_=target_context('B')
    event('stage_started',stage=tag,recipe=recipe,validation_feedback=False)
    started=time.monotonic();report=None
    if recipe['kind']=='reference_tree':
        old=a['old_metadata'];params=dict(old['definition']['params'])
        params['n_estimators']=int(old['best_iteration'])
        checkpoint=EXP/(tag+'.txt');archive_partial(checkpoint)
        scored=UOUT[fit_ids]==0
        learner=lgb.LGBMRegressor(**params,objective='regression_l1',subsample=.9,subsample_freq=1,
            colsample_bytree=.9,n_jobs=THREADS,random_state=recipe['seed'],deterministic=True,force_col_wise=True,verbosity=-1)
        learner.fit(X[fit_ids][scored],Y[fit_ids][scored],feature_name=names)
        learner.booster_.save_model(str(checkpoint));del learner;gc.collect()
        report=dict(validation_used=False,frozen_trees=params['n_estimators'],selected_using='A only')
        epoch=None
    else:
        checkpoint=EXP/(tag+'.pt');epoch_log=EXP/(tag+'_epochs.jsonl');archive_partial(checkpoint,epoch_log)
        start=None
        if recipe['kind']=='warm':
            base=fit_B(recipe['base']);assert base['fit_group_sha256']==design['group_hashes']['B_fit']
            start=base['checkpoint_path']
        report=fit_fixed(X[fit_ids],Y[fit_ids],UOUT[fit_ids],recipe['model_spec'],recipe['train_cfg'],
            recipe['seed'],a['selected_lr_schedule'],checkpoint,start_checkpoint=start,epoch_log_path=epoch_log)
        epoch=report['executed_epochs']
    r=dict(name=name,split='B',kind=recipe['kind'],recipe=recipe,identity=identity,
        checkpoint_path=str(checkpoint),checkpoint_sha256=digest(checkpoint),report=report,
        best_epoch=epoch,selected_lr_schedule=a.get('selected_lr_schedule'),
        training_breaths=len(fit_ids),parameter_count=a.get('parameter_count'),
        fit_group_sha256=design['group_hashes']['B_fit'])
    prediction=infer_record(r,X[val_ids])[mask]
    score,pred_hash=save_score(tag,prediction,'B')
    r.update(validation_mae=score,prediction_file=str(EXP/(tag+'_predictions.npy')),
             prediction_sha256=pred_hash,elapsed_seconds=time.monotonic()-started)
    write_json(path,r);refresh_results();B_RECORDS[name]=r
    event('stage_completed',stage=tag,validation_mae=score,seconds=r['elapsed_seconds'])
    gc.collect();torch.cuda.empty_cache();return r

def paired_interval(first,second,split='B',repeats=1000):
    ids,mask,y,_=target_context(split);counts=mask.sum(axis=1)
    assert (counts>0).all()
    diff=np.abs(first-y)-np.abs(second-y);starts=np.r_[0,np.cumsum(counts)[:-1]]
    sums=np.add.reduceat(diff,starts);rng=np.random.default_rng(20261206);draws=[]
    for offset in range(0,repeats,100):
        take=rng.integers(0,len(counts),size=(min(100,repeats-offset),len(counts)))
        draws.extend((sums[take].sum(axis=1)/counts[take].sum(axis=1)).tolist())
    low,high=np.quantile(draws,[.025,.975])
    return dict(difference_first_minus_second=float(diff.mean()),bootstrap_95_low=float(low),
        bootstrap_95_high=float(high),bootstrap_unit='whole breath',repeats=repeats)

if RUN:
    B_RECORDS={}
    required=list(dict.fromkeys(['gru_s20261102',frozen['best_individual']]+frozen['pair_members']))
    for name in required:fit_B(name)
    def blended(split,members,weights):
        records=A_RECORDS if split=='A' else B_RECORDS
        return sum(w*np.load(records[n]['prediction_file']) for n,w in zip(members,weights))
    _,_,b_y,_=target_context('B')
    a_ref=A_RECORDS['gru_s20261102'];b_ref=B_RECORDS['gru_s20261102']
    a_ind=A_RECORDS[frozen['best_individual']];b_ind=B_RECORDS[frozen['best_individual']]
    b_pair_pred=blended('B',frozen['pair_members'],frozen['pair_weights'])
    b_pair=float(np.abs(b_pair_pred-b_y).mean())
    pair_A=float(frozen['pair_A_mae'])
    table=pd.DataFrame([
      dict(approach='reference',members=['gru_s20261102'],weights=[1.],A_mae=a_ref['validation_mae'],B_mae=b_ref['validation_mae'],eligible=True),
      dict(approach='best_individual',members=[a_ind['name']],weights=[1.],A_mae=a_ind['validation_mae'],B_mae=b_ind['validation_mae'],
        eligible=a_ind['validation_mae']<a_ref['validation_mae']-MIN_GAIN and b_ind['validation_mae']<b_ref['validation_mae']-MIN_GAIN),
      dict(approach='error_diverse_pair',members=frozen['pair_members'],weights=frozen['pair_weights'],A_mae=pair_A,B_mae=b_pair,
        eligible=pair_A<a_ind['validation_mae']-MIN_GAIN and b_pair<b_ind['validation_mae']-MIN_GAIN and b_pair<b_ref['validation_mae']-MIN_GAIN)])
    table.to_csv(EXP/'approach_comparison.csv',index=False)
    chosen=table[table.eligible].sort_values(['A_mae','approach']).iloc[0].to_dict()
    evidence=dict(**chosen,selection_rule=PLAN['B_rule'],B_weights_tuned=False,
        A_training_breaths=61114,B_training_breaths=54323,A_validation_breaths=6791,B_validation_breaths=6791,
        reference_A_mae=a_ref['validation_mae'],reference_B_mae=b_ref['validation_mae'],
        paired_individual_vs_reference=paired_interval(np.load(b_ind['prediction_file']),np.load(b_ref['prediction_file'])),
        paired_pair_vs_individual=paired_interval(b_pair_pred,np.load(b_ind['prediction_file'])),
        evidence_is_from_validation_models=True,final_refit_has_independent_holdout=False)
    # Convert dataframe/numpy scalar types before JSON serialization.
    evidence=json.loads(json.dumps(evidence,default=lambda x:x.item() if hasattr(x,'item') else x))
    write_json(EXP/'validation_selection.json',evidence);event('validation_selection_completed',selection=evidence)
    display(table)

## Final all-labeled fit — separate from validation evidence

The user explicitly requested this stage after selecting architecture, duration
and weights. Each selected model is freshly fitted on all 67,905 approved labeled
breaths, with its own all-training statistics and A-frozen schedule. Warm recipes
first train a fresh all-data base, then reset Adam for their frozen continuation
while retaining that base's coordinates. A/B validation checkpoints stay intact.
The final model has no independent A/B holdout and does not inherit their scores.

In [ ]:
def final_fit(name):
    if name in FINAL_RECORDS:return FINAL_RECORDS[name]
    a=A_RECORDS[name];recipe=a['recipe'];path=FINAL/('record_'+name+'.json')
    identity=signature(dict(recipe=recipe,A_hash=a['checkpoint_sha256'],rates=a.get('selected_lr_schedule'),
                            all_train_hash=hashlib.sha256(breath_ids.tobytes()).hexdigest(),design=signature(design)))
    if path.exists():
        r=json.loads(path.read_text());assert r['identity']==identity
        assert digest(Path(r['checkpoint_path']))==r['checkpoint_sha256'];FINAL_RECORDS[name]=r;return r
    event('final_fit_started',member=name,training_breaths=len(ALL),validation_used=False)
    started=time.monotonic();report=None
    if recipe['kind']=='reference_tree':
        params=dict(a['old_metadata']['definition']['params']);params['n_estimators']=int(a['old_metadata']['best_iteration'])
        checkpoint=FINAL/(name+'.txt');archive_partial(checkpoint)
        mask=UOUT==0
        learner=lgb.LGBMRegressor(**params,objective='regression_l1',subsample=.9,subsample_freq=1,colsample_bytree=.9,
            n_jobs=THREADS,random_state=recipe['seed'],deterministic=True,force_col_wise=True,verbosity=-1)
        learner.fit(X[mask],Y[mask],feature_name=names);learner.booster_.save_model(str(checkpoint));del learner;gc.collect()
        report=dict(validation_used=False,frozen_trees=params['n_estimators'])
    else:
        checkpoint=FINAL/(name+'.pt');epoch_log=FINAL/(name+'_epochs.jsonl');archive_partial(checkpoint,epoch_log)
        start=final_fit(recipe['base'])['checkpoint_path'] if recipe['kind']=='warm' else None
        rates=a['selected_lr_schedule']
        assert rates or recipe['kind']=='warm'
        report=fit_fixed(X,Y,UOUT,recipe['model_spec'],recipe['train_cfg'],recipe['seed'],rates,
            checkpoint,start_checkpoint=start,epoch_log_path=epoch_log)
    r=dict(name=name,kind=recipe['kind'],recipe=recipe,identity=identity,checkpoint_path=str(checkpoint),
        checkpoint_sha256=digest(checkpoint),training_breaths=67905,scored_training_rows=int((UOUT==0).sum()),
        validation_used=False,has_independent_holdout=False,report=report,elapsed_seconds=time.monotonic()-started)
    write_json(path,r);FINAL_RECORDS[name]=r
    event('final_fit_completed',member=name,seconds=r['elapsed_seconds'],checkpoint_sha256=r['checkpoint_sha256'])
    gc.collect();torch.cuda.empty_cache();return r

if RUN:
    FINAL.mkdir(parents=True,exist_ok=True);FINAL_RECORDS={}
    selection=json.loads((EXP/'validation_selection.json').read_text())
    selection_hash=digest(EXP/'validation_selection.json')
    for name in selection['members']:final_fit(name)
    assert digest(DATA/'test.csv')==quality['files']['test.csv']['sha256']
    assert digest(DATA/'sample_submission.csv')=='de4c137a22804654b597723f32470b24f23d9ef662205eca0fe31b68970f0594'
    test=pd.read_csv(DATA/'test.csv',dtype=DTYPES)
    xt,test_names,test_breaths=features(test)
    assert test_names==names and not np.intersect1d(test_breaths,breath_ids).size
    xt=xt.reshape(-1,80,20);prediction=np.zeros(len(test),dtype=np.float64)
    for name,weight in zip(selection['members'],selection['weights']):
        r=FINAL_RECORDS[name];assert digest(Path(r['checkpoint_path']))==r['checkpoint_sha256']
        prediction+=weight*infer_record(r,xt).ravel().astype(np.float64)
    sample=pd.read_csv(DATA/'sample_submission.csv')
    assert test.id.is_unique and sample.id.is_unique and set(test.id)==set(sample.id)
    ordered=pd.Series(prediction,index=test.id.to_numpy()).reindex(sample.id).to_numpy()
    assert len(ordered)==603600 and np.isfinite(ordered).all()
    submission=pd.DataFrame({'id':sample.id.to_numpy(),'pressure':ordered})
    submission.to_csv(FINAL/'submission.csv',index=False,float_format='%.8f')
    check=pd.read_csv(FINAL/'submission.csv')
    assert list(check.columns)==['id','pressure'] and np.array_equal(check.id.to_numpy(),sample.id.to_numpy())
    assert len(check)==603600 and np.isfinite(check.pressure).all()
    manifest=dict(training_group='all_approved_labeled',training_breaths=67905,
        scored_training_points=int((UOUT==0).sum()),selected_members=selection['members'],weights=selection['weights'],
        model_files=[FINAL_RECORDS[n]['checkpoint_path'] for n in selection['members']],
        model_hashes=[FINAL_RECORDS[n]['checkpoint_sha256'] for n in selection['members']],
        recipes={n:FINAL_RECORDS[n]['recipe'] for n in selection['members']},
        validation_evidence_file=str(EXP/'validation_selection.json'),validation_evidence_sha256=selection_hash,
        validation_scores_are_from_prior_validation_models=True,final_refit_has_independent_holdout=False,
        submission_file=str(FINAL/'submission.csv'),submission_rows=len(check),submission_sha256=digest(FINAL/'submission.csv'),
        train_sha256=digest(DATA/'train.csv'),test_sha256=digest(DATA/'test.csv'),uploaded=False)
    write_json(FINAL/'final_manifest.json',manifest)
    event('FINAL_CSV_VERIFIED',rows=len(check),sha256=manifest['submission_sha256'],training_breaths=67905)
    print('Final all-labeled CSV:',FINAL/'submission.csv')
    display(manifest)

## Review saved outcomes

Run with the default flags to review without training. The final CSV's model
uses all approved labels, while the A/B scores below belong to earlier held-out
models. Keep those distinctions when writing the assignment report.

In [ ]:
if (EXP/'results.csv').exists():display(pd.read_csv(EXP/'results.csv'))
else:print('Prepared refinement plan; no v004 training results yet.')
if (EXP/'approach_comparison.csv').exists():display(pd.read_csv(EXP/'approach_comparison.csv'))
if (FINAL/'final_manifest.json').exists():display(json.loads((FINAL/'final_manifest.json').read_text()))